In [6]:
# Import required libraries
import os
import numpy as np
from PIL import Image
import torch
from transformers import AutoModel
from torchvision import transforms
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load TIPSv2 model
print("Loading TIPSv2 model...")
model = AutoModel.from_pretrained("google/tipsv2-b14", trust_remote_code=True, dtype="auto").to(device)
print("Model loaded successfully!")

# Setup image transform for TIPSv2
transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

Using device: cuda
Loading TIPSv2 model...
Model loaded successfully!


# Image Search Engine using TIPSv2

This notebook creates an image search engine that allows you to search for similar images in the Data folder using TIPSv2. TIPSv2 is a powerful vision transformer that captures detailed visual embeddings of images, enabling efficient visual similarity search.

In [7]:
# Load all image paths from the Data folder
data_folder = Path("Data")
image_files = sorted(list(data_folder.glob("*.jpg")))
print(f"Found {len(image_files)} images in the Data folder")

# Load and prepare images
print("Loading images...")
images = []
image_paths = []
image_tensors = []

for img_path in image_files:
    try:
        img = Image.open(img_path).convert("RGB")
        images.append(img)
        image_paths.append(str(img_path))
        
        # Prepare tensor for model
        img_tensor = transform(img).unsqueeze(0)
        image_tensors.append(img_tensor)
    except Exception as e:
        print(f"Error loading {img_path}: {e}")

Found 126 images in the Data folder
Loading images...
Error loading Data\Q151047.jpg: Image size (565200000 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.
Error loading Data\Q208758.jpg: Image size (781950000 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.
Error loading Data\Q23922.jpg: Image size (229533750 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.
Error loading Data\Q45585.jpg: Image size (712680000 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.
Error loading Data\Q59815.jpg: Image size (298581143 pixels) exceeds limit of 178956970 pixels, could be decompression bomb DOS attack.


In [13]:
cache_path = "Data/embeddings_cache.pt"
image_embeddings = []
with torch.no_grad():
    for img_tensor in image_tensors:
        img_tensor = img_tensor.to(device)
        # Use the same high-level encode_image function for consistency
        embedding_output = model.encode_image(img_tensor)
        # The correct embedding is in the 'cls_token' attribute
        embedding = embedding_output.cls_token
        # Squeeze the tensor to remove unnecessary dimensions before converting to numpy
        image_embeddings.append(embedding.squeeze().cpu().numpy())

# Save embeddings to cache
print(f"Saving {len(image_embeddings)} embeddings to cache...")
torch.save({
    "embeddings": image_embeddings,
    "image_paths": image_paths
}, cache_path)
print("Embeddings cached successfully!")

Saving 121 embeddings to cache...
Embeddings cached successfully!


In [14]:
# Check the dimensions of the embeddings
if 'image_embeddings' in locals() and image_embeddings:
    # Check the shape of the first embedding in the list
    first_embedding_shape = image_embeddings[0].shape
    print(f"Shape of a single image embedding: {first_embedding_shape}")
    
    # Check the shape of the entire embeddings tensor when stacked
    embeddings_tensor = torch.tensor(np.array(image_embeddings))
    if embeddings_tensor.ndim == 3 and embeddings_tensor.shape[1] == 1:
        embeddings_tensor = embeddings_tensor.squeeze(1)
        
    print(f"Shape of the full embeddings tensor for all {len(image_embeddings)} images: {embeddings_tensor.shape}")
else:
    print("Embeddings have not been generated yet. Please run the previous cell.")

Shape of a single image embedding: (768,)
Shape of the full embeddings tensor for all 121 images: torch.Size([121, 768])
